# 23 System test photos (copy candidates)

Copies a small set of candidate photos for the system tests (Table 5.29: T11 to T19, T24) into
`CropDiseaseDiagnosis/system_test_photos/` on Drive, with `candidates.csv`.

Nothing is trained or scored here. Download the folder (right-click, Download) and give it to
the laptop: the deployed pipeline is run there on every candidate, and the photos for each test
are chosen and their expected results written down **before** the tests are run.

Groups:
- `test_tomato_late_blight`, `test_potato_late_blight`: the PlantVillage test partition (T11, T24).
- `test_mixed`: other test-partition photos, for the batch of five (T12).
- `test_maize`: maize test photos, for "Not sure / mixed" (T18).
- `selection`: field photos from the selection set, to find one below the threshold (T17).

In [ ]:
from pathlib import Path
import sys
from google.colab import drive

drive.mount("/content/drive")
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")
ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))
import config

In [ ]:
import shutil
import pandas as pd

SEED = config.RANDOM_SEED
OUT = config.DATA_ROOT / "system_test_photos"
OUT.mkdir(parents=True, exist_ok=True)


def resolve(p):
    p = Path(str(p))
    return p if p.is_absolute() else config.RAW_DATA_DIR / p  # split paths are relative to datasets/raw


test = pd.read_csv(config.SPLITS_DIR / "test.csv")
selection = pd.read_csv(config.SPLITS_DIR / "selection.csv")
print("test:", len(test), "photos; selection:", len(selection), "photos")
print("columns:", list(test.columns))


def pick(df, mask, n, group):
    part = df[mask]
    take = part.sample(n=min(n, len(part)), random_state=SEED)
    return take.assign(group=group)


groups = [
    pick(test, test["class"] == "Tomato___Late_blight", 8, "test_tomato_late_blight"),
    pick(test, test["class"] == "Potato___Late_blight", 8, "test_potato_late_blight"),
    pick(test, ~test["class"].str.contains("Late_blight") & ~test["class"].str.startswith("Corn"), 10, "test_mixed"),
    pick(test, test["class"].str.startswith("Corn"), 8, "test_maize"),
    pick(selection, selection["class"].notna(), 40, "selection"),
]
chosen = pd.concat(groups, ignore_index=True)
rows = []
for i, r in chosen.iterrows():
    src = resolve(r["path"])
    folder = OUT / r["group"]
    folder.mkdir(exist_ok=True)
    name = f"{i:03d}_{Path(src).name}"
    shutil.copy(src, folder / name)
    rows.append({"file": f"{r['group']}/{name}", "group": r["group"], "class": r["class"],
                 "source": r.get("source", ""), "original_path": str(r["path"])})
pd.DataFrame(rows).to_csv(OUT / "candidates.csv", index=False)
print(len(rows), "photos copied to", OUT)
display(pd.DataFrame(rows).groupby("group").size())